# BadBlueprint full scoring (Colab)
This notebook runs the BadBlueprint full scoring flow using the open-weight model and the vendored harness.

## A. Runtime / GPU check

In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

print("Python:", platform.python_version())
run("nvidia-smi || true")
run("python - <<'PY'
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
PY")
run("free -h")
run("df -h /")

if shutil.which("nvidia-smi") is None:
    print("WARNING: No GPU detected. Full scoring may be extremely slow.")

## B. Environment setup

In [ ]:
import subprocess


def pip_install(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Uninstall potentially conflicting packages (safe no-op if missing)
for pkg in ["torchvision", "torchaudio"]:
    pip_install(f"pip uninstall -y {pkg} || true")

pip_install("pip install -U --quiet git+https://github.com/huggingface/transformers.git")
pip_install("pip install -U --quiet accelerate safetensors huggingface_hub uvicorn fastapi requests")

## C. Model download

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os

model_id = "openai/gpt-oss-20b"
local_dir = Path("/content/models/gpt-oss-20b")
local_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")

snapshot_download(
    repo_id=model_id,
    local_dir=str(local_dir),
    local_dir_use_symlinks=False,
    token=hf_token,
)

file_count = sum(1 for _ in local_dir.rglob("*"))
print(f"Model downloaded to: {local_dir} (files: {file_count})")

## D. Start local OpenAI-compatible endpoint

In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

server_log = Path("/content/transformers_server.log")
server_pid = Path("/content/transformers_server.pid")

if server_pid.exists():
    print("Server appears to be running already. Skipping start.")
else:
    cmd = [
        "transformers",
        "serve",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--model",
        "/content/models/gpt-oss-20b",
    ]
    print("Starting server:", " ".join(cmd))
    with server_log.open("w") as log_f:
        proc = subprocess.Popen(
            cmd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,
        )
    server_pid.write_text(str(proc.pid))
    time.sleep(5)
    print(f"Server PID: {proc.pid}")

### Healthcheck

In [ ]:
import time
import requests

endpoint = "http://127.0.0.1:8000/v1"

ok = False
last_err = None
for _ in range(20):
    try:
        resp = requests.get(f"{endpoint}/models", timeout=5)
        if resp.status_code == 200:
            print("Server is healthy.")
            print(resp.json())
            ok = True
            break
        else:
            last_err = f"HTTP {resp.status_code}: {resp.text[:200]}"
    except Exception as exc:
        last_err = str(exc)
    time.sleep(3)

if not ok:
    raise RuntimeError(f"Server healthcheck failed: {last_err}")

## E. Clone repo and prepare submission bundle

In [ ]:
import os
import subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

repo_dir = Path.cwd()
if not (repo_dir / "scripts" / "export_badblueprint_submission.py").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")
    if not repo_dir.exists():
        run("git clone https://github.com/Purple-Vanguard/purple-vanguard-scenarios.git", cwd=Path("/content"))

run("python scripts/export_badblueprint_submission.py", cwd=repo_dir)
run("python scripts/validate_submission_bundle.py submissions/purple_vanguard/badblueprint", cwd=repo_dir)

## F. Install vendored harness

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import re
import tomllib

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

subprocess.run("pip install -e vendor/agentbeats-lambda", shell=True, check=True, cwd=repo_dir)

pyproject = repo_dir / "vendor" / "agentbeats-lambda" / "pyproject.toml"
cli_name = None
if pyproject.exists():
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = sorted(scripts.keys())[0]

if cli_name is None:
    # Fallback: list bin scripts
    bin_dir = Path(sys.executable).parent
    candidates = [p.name for p in bin_dir.iterdir() if p.is_file() and "agent" in p.name]
    cli_name = candidates[0] if candidates else None

if not cli_name:
    raise RuntimeError("Could not detect harness CLI entrypoint.")

print("Detected harness CLI:", cli_name)

## G. Configure harness to use local endpoint

In [ ]:
import os
import re
import subprocess
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

search_patterns = "BASE_URL|API_BASE|OPENAI_|endpoint|/v1/chat/completions|/v1/models"
rg_cmd = ["rg", "-n", search_patterns, str(repo_dir / "vendor" / "agentbeats-lambda")]
result = subprocess.run(rg_cmd, check=False, capture_output=True, text=True)
print(result.stdout)

env_names = set()
for line in result.stdout.splitlines():
    for match in re.findall(r"os\.environ\["([A-Z0-9_]+)"\]", line):
        env_names.add(match)
    for match in re.findall(r"getenv\("([A-Z0-9_]+)"\)", line):
        env_names.add(match)

if not env_names:
    raise RuntimeError("No environment variables found for endpoint configuration.")

endpoint = "http://127.0.0.1:8000/v1"

for name in sorted(env_names):
    if any(key in name for key in ["BASE", "ENDPOINT", "URL", "HOST"]):
        os.environ[name] = endpoint
    if "API_KEY" in name:
        os.environ.setdefault(name, "DUMMY_KEY")

os.environ["MODEL_NAME"] = os.environ.get("MODEL_NAME", "gpt-oss-20b")

print("Configured endpoint-related environment variables:", ", ".join(sorted(env_names)))
print("Model name set to:", os.environ["MODEL_NAME"])

## H. Run FULL SCORING

In [ ]:
import os
import subprocess
from pathlib import Path
import sys

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / "full_score.log"

pyproject = repo_dir / "vendor" / "agentbeats-lambda" / "pyproject.toml"
cli_name = None
if pyproject.exists():
    import tomllib
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = sorted(scripts.keys())[0]

if not cli_name:
    raise RuntimeError("Could not detect harness CLI entrypoint.")

cmd = [cli_name, "score", "--mode", "full", "submissions/purple_vanguard/badblueprint/scenario_badblueprint.toml"]

print("Running:", " ".join(cmd))
with log_path.open("w") as log_f:
    proc = subprocess.run(cmd, cwd=repo_dir, stdout=log_f, stderr=subprocess.STDOUT)

exit_code = proc.returncode
print("Scoring exit code:", exit_code)

# Collect agent cards if available
for card in repo_dir.rglob("agent-card-*.json"):
    target = results_dir / card.name
    if card != target:
        target.write_text(card.read_text())

# Ensure standardized names
name_map = {
    "agent-card-green.json": "agent-card-green.json",
    "agent-card-attacker.json": "agent-card-attacker.json",
    "agent-card-defender.json": "agent-card-defender.json",
}
for src_name, dst_name in name_map.items():
    src = results_dir / src_name
    if src.exists():
        (results_dir / dst_name).write_text(src.read_text())

exit_code

## I. Write score_status.json

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
results_dir.mkdir(parents=True, exist_ok=True)

pin_path = repo_dir / "vendor" / "agentbeats-lambda" / "COMMIT_PIN.txt"
commit_pin = pin_path.read_text().strip() if pin_path.exists() else "unknown"

log_path = results_dir / "full_score.log"

success = log_path.exists() and log_path.stat().st_size > 0
notes = "ok" if success else "scoring failed or produced empty log"

status = {
    "mode": "full",
    "ran_at": datetime.now(timezone.utc).isoformat(),
    "submission_path": "submissions/purple_vanguard/badblueprint",
    "harness_commit_pin": commit_pin,
    "model_endpoint": "http://127.0.0.1:8000/v1",
    "success": bool(success),
    "notes": notes,
}

(results_dir / "score_status.json").write_text(json.dumps(status, indent=2, sort_keys=True))
print("Wrote", results_dir / "score_status.json")

## J. Package results for download

In [ ]:
import os
import tarfile
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "results" / "badblueprint").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
archive_path = repo_dir / "results_badblueprint_colab.tgz"

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(results_dir, arcname="badblueprint")

size = archive_path.stat().st_size
print(f"Archive created: {archive_path} ({size} bytes)")

## Cleanup (stop server)

In [ ]:
import os
import signal
from pathlib import Path

server_pid = Path("/content/transformers_server.pid")
if server_pid.exists():
    pid = int(server_pid.read_text())
    try:
        os.killpg(pid, signal.SIGTERM)
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    server_pid.unlink(missing_ok=True)
else:
    print("No server PID file found.")